<a href="https://colab.research.google.com/github/sumair789-lgtm/urdu-ocr-codesaviours-si26--Sumair-/blob/main/SI26-Week4-Sumair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Week 4 Tasks

Install Library


In [ ]:
!pip install transformers torch pillow pandas sentencepiece -q

Mount Drive

In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/urdu-ocr-si26'
csv_path = os.path.join(base_path, 'data', 'labels.csv')
print('Base path exists:', os.path.exists(base_path))
print('CSV path exists:', os.path.exists(csv_path))

Mounted at /content/drive
Base path exists: True
CSV path exists: True


Dataset Class

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, max_target_length=128):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.max_target_length = max_target_length
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True
        ).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

Build processor + dataset (verified path baked in)

In [ ]:
from transformers import TrOCRProcessor, ViTImageProcessor, RobertaTokenizer
import os

# Same method as your Week 3 notebook — avoids the sentencepiece/fast-tokenizer error
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed')
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

csv_path = '/content/drive/MyDrive/urdu-ocr-si26/data/labels.csv'
dataset = UrduOCRDataset(csv_path, processor)

if 'Unnamed: 1' in dataset.data.columns:
    dataset.data = dataset.data.rename(columns={'Unnamed: 1': 'text'})

drive_base_path = '/content/drive/MyDrive/urdu-ocr-si26'
dataset.data['image'] = dataset.data['image'].apply(
    lambda x: os.path.join(drive_base_path, x) if not str(x).startswith('/content') else x
)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset loaded: 273 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!


Train/Test Split

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(f'Total samples: {len(dataset)}')
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Total samples: 273
Training samples: 218
Testing samples: 55


Load Model

In [ ]:
from transformers import VisionEncoderDecoderModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda


config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


Training Setup

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 55
Ready to train!


Loop

In [ ]:
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/20
------------------------------
  Batch 0/55 | Loss: 17.4326
  Batch 10/55 | Loss: 5.2601
  Batch 20/55 | Loss: 4.3983
  Batch 30/55 | Loss: 3.9415
  Batch 40/55 | Loss: 3.6186
  Batch 50/55 | Loss: 3.6549
Epoch 1 complete | Average Loss: 4.8601

Epoch 2/20
------------------------------
  Batch 0/55 | Loss: 3.5866
  Batch 10/55 | Loss: 3.4754
  Batch 20/55 | Loss: 3.4583
  Batch 30/55 | Loss: 3.3682
  Batch 40/55 | Loss: 3.4727
  Batch 50/55 | Loss: 3.4079
Epoch 2 complete | Average Loss: 3.5733

Epoch 3/20
------------------------------
  Batch 0/55 | Loss: 3.3881
  Batch 10/55 | Loss: 3.5830
  Batch 20/55 | Loss: 3.4878
  Batch 30/55 | Loss: 3.5738
  Batch 40/55 | Loss: 3.8978
  Batch 50/55 | Loss: 3.6941
Epoch 3 complete | Average Loss: 3.5225

Epoch 4/20
------------------------------
  Batch 0/55 | Loss: 3.4939
  Batch 10/55 | Loss: 3.6682
  Batch 20/55 | Loss: 3.4985
  Batch 30/55 | Loss: 3.4541
  Batch 40/55 | Loss: 3.4384
  Batch 50/55 | Loss: 3.4258
Epoch 4 complet

Evaluation

In [ ]:
model.eval()
print('=== Model Evaluation on Test Images ===\n')

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual: {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted: ن�����������������
Actual: پاکستانی اور انڈین شخصیات کے درمیان ٹریک ٹو ملاقاتوں اور وزرائے اعظم کو لکھے گئے خط کے حوالے سے

Predicted: ��������������������
Actual: میں سے کسی نے پولیس پر فائرنگ کی۔ انہوں

Predicted: �ی���ااااا�������
Actual: اتنے میں وہ تینوں ہال میں داخل ہوئے ۔ مہمانوں نے تالیاں بجائیں جن کا جواب تینوں نے یوں ہاتھ ہلا ہلا �

Predicted: ا�������������������
Actual: اپنے مظاہروں کا سلسلہ ختم کر دیں۔ انہوں نے تمام تنظیمی ونگز

Predicted: ووو�    ����������
Actual: محب ایک دفعہ ایک عورت کا بچہ گم ہو گیا۔ وہ اسے قافلے میں ڈھونڈتی پھر رہی تھی۔ وہ ایک ایک

Predicted: ������������������
Actual: بادشاہوں نے اپنے اقتدار کو جائز قرار دینے اور اسے دوام بخشنے

Predicted: �ااااااااااا������
Actual: مظفر الیکٹرک اینڈ الیکٹرانکس سینٹر

Predicted: �ی���اااا��������
Actual: تعلیم ہر انسان کا حق ہے

Predicted: ������������������
Actual: حج کی ادائیگی کا طری آٹھ ذوالحجہ سے بارہ ذوالحجہ تک حج کے مخصوص پانچ دن ہیں۔ ان پانچ دنوں میں ح

Pre

In [ ]:
save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')
print('You can load this model again next week without retraining')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining


In [ ]:
print("=== Raw label check (first 3 rows) ===")
for i in range(3):
    print(f"Label {i}: {dataset.data.iloc[i]['text']}")

=== Raw label check (first 3 rows) ===
Label 0: بیرون ملک مقیم پاکستانیوں کی بہبود کیلئے جاری منصوبوں میں مزید تیزی لائی جائے، وزیر اعظم
Label 1: نجکاری کمیشن نے فیسکو اور گیپکو کے لیے اظہار دلچسپی جمع کرانے کی تاریخ میں توسیع کر دی
Label 2: رولیکس نے 1940 کی دہائی میں 'رولیکس 4113' ماڈل کی صرف 12 گھڑیاں ہی بنائی تھیں جن میں سےاب صرف 9 گھڑیاں ہی دنیا میں


In [ ]:
!pip install jiwer -q
from jiwer import cer

model.eval()
predicted_texts, actual_texts = [], []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        generated_ids = model.generate(pixel_values)
        predicted_texts.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
        actual_texts.extend(processor.batch_decode(labels, skip_special_tokens=True))

score = cer(actual_texts, predicted_texts)
print(f'Character Error Rate (CER): {score:.4f} ({score*100:.1f}%)')
print("\n=== Sample comparison ===")
for i in range(5):
    print(f'Predicted: {predicted_texts[i]}')
    print(f'Actual: {actual_texts[i]}')
    print()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Character Error Rate (CER): 1.0012 (100.1%)

=== Sample comparison ===
Predicted: SALES:
Actual: مسلمان بھی اللہ تعالیٰ محبوب اور نبی آخرالزمان حضرت

Predicted: CSTALL:$168.00
Actual: مال اور طولِ عمر کی حرص حضرت انس رضی اللہ عنہ نے فرمایا نبی کریم صلی اللہ علیہ وسلم

Predicted: =
Actual: رولیکس نے 1940 کی دہائی میں 'رولیکس 4113' ماڈل کی صرف 12 گھڑیاں ہی بنائی تھیں جن میں سےاب صرف 9 گھڑیاں ہ

Predicted: EXCLUDING AND EXCLUDES ON BACK
Actual: ایف آئی آر میں کیا لکھا ہے؟

Predicted: CARD SALAD
Actual: دَرَخْت - بُلْبُل - اُداس - جُگْنُو - رَوشْنِی



Loaded your 40-epoch model from Drive

In [ ]:
import os
import torch
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/urdu-ocr-si26'
csv_path = os.path.join(base_path, 'data', 'labels.csv')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

from transformers import VisionEncoderDecoderModel, TrOCRProcessor

save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
processor = TrOCRProcessor.from_pretrained(save_path)
model = VisionEncoderDecoderModel.from_pretrained(save_path)
model = model.to(device)

print('Loaded your 40-epoch model from Drive')

Using device: cuda


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Loaded your 40-epoch model from Drive


In [ ]:
backup_path = '/content/drive/MyDrive/SI26-urdu-ocr-model-40epoch-backup'
model.save_pretrained(backup_path)
processor.save_pretrained(backup_path)
print(f'Backed up to: {backup_path}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Backed up to: /content/drive/MyDrive/SI26-urdu-ocr-model-40epoch-backup


In [ ]:
all_text = ''.join(dataset.data['text'].astype(str))
unique_chars = sorted(set(all_text))

tokens_to_add = []
for ch in unique_chars:
    if ch.strip() == '':
        continue
    ids = processor.tokenizer.encode(ch, add_special_tokens=False)
    if len(ids) > 1:
        tokens_to_add.append(ch)

print(f'Total unique characters: {len(unique_chars)}')
print(f'Fragmented characters (ab fix honge): {len(tokens_to_add)}')

num_added = processor.tokenizer.add_tokens(tokens_to_add)
print(f'\nAdded {num_added} new tokens')
print(f'New vocab size: {len(processor.tokenizer)}')

model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.config.decoder.vocab_size = len(processor.tokenizer)
model.config.vocab_size = len(processor.tokenizer)
print('Model resized successfully')

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Total unique characters: 93
Fragmented characters (ab fix honge): 52

Added 52 new tokens
New vocab size: 50317
Model resized successfully


In [ ]:
from torch.utils.data import random_split, DataLoader

dataset = UrduOCRDataset(csv_path, processor)
if 'Unnamed: 1' in dataset.data.columns:
    dataset.data = dataset.data.rename(columns={'Unnamed: 1': 'text'})
dataset.data['image'] = dataset.data['image'].apply(
    lambda x: os.path.join(base_path, x) if not str(x).startswith('/content') else x
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

sample = dataset[0]
print(f'Sample label length now: {len(sample["labels"])}')
print(f'Train: {train_size} | Test: {test_size}')

Dataset loaded: 273 samples
Sample label length now: 128
Train: 218 | Test: 55


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch + 1} complete | Average Loss: {total_loss/len(train_loader):.4f}')

print('\nTraining complete!')


Epoch 1/10
------------------------------
  Batch 0/55 | Loss: 9.0001
  Batch 10/55 | Loss: 6.1033
  Batch 20/55 | Loss: 5.9756
  Batch 30/55 | Loss: 5.9027
  Batch 40/55 | Loss: 5.2853
  Batch 50/55 | Loss: 4.5561
Epoch 1 complete | Average Loss: 6.0350

Epoch 2/10
------------------------------
  Batch 0/55 | Loss: 4.4800
  Batch 10/55 | Loss: 3.9667
  Batch 20/55 | Loss: 3.5569
  Batch 30/55 | Loss: 3.3725
  Batch 40/55 | Loss: 3.4913
  Batch 50/55 | Loss: 3.3881
Epoch 2 complete | Average Loss: 3.6694

Epoch 3/10
------------------------------
  Batch 0/55 | Loss: 3.2413
  Batch 10/55 | Loss: 3.6825
  Batch 20/55 | Loss: 3.6448
  Batch 30/55 | Loss: 3.4331
  Batch 40/55 | Loss: 3.3389
  Batch 50/55 | Loss: 3.2714
Epoch 3 complete | Average Loss: 3.3305

Epoch 4/10
------------------------------
  Batch 0/55 | Loss: 3.5208
  Batch 10/55 | Loss: 3.0950
  Batch 20/55 | Loss: 3.5612
  Batch 30/55 | Loss: 3.1699
  Batch 40/55 | Loss: 3.1422
  Batch 50/55 | Loss: 3.2615
Epoch 4 complete

In [ ]:
model.eval()
correct = 0
total = 0
predicted_texts, actual_texts = [], []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)
        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            predicted_texts.append(pred)
            actual_texts.append(actual)

print(f'Accuracy: {(correct/total)*100:.1f}% ({correct}/{total})')
for i in range(5):
    print(f'\nPredicted: {predicted_texts[i]}')
    print(f'Actual: {actual_texts[i]}')

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Accuracy: 0.0% (0/55)

Predicted: یییییییییییییییییییی
Actual: کو نہ صرف اپنے مفادات کیلئے استعمال کر رہے ہیں بلکہ معصوم

Predicted:                     
Actual: جواب میں مظاہرین نے پولیس پر پتھراؤ شروع کر دیا۔ بنوں کے

Predicted: اااااااااااااااااااا
Actual: پاکستانی اور انڈین شخصیات کے درمیان ٹریک ٹو ملاقاتوں اور وزرائے اعظم کو لکھے گئے خط کے حوالے سے

Predicted: اااااااااااااااااااا
Actual: والدین کی عزت کرنی چاہیے

Predicted: اااااااااااااااااااا
Actual: فرانس نے منگولیا کو 7 کروڑ سال پرانا ڈائناسور کا ڈھانچہ واپس کر دیا


Evaluation

In [ ]:
model.eval()
correct = 0
total = 0
predicted_texts, actual_texts = [], []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        generated_ids = model.generate(pixel_values, max_new_tokens=128)  # fixed: match training length
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)
        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            predicted_texts.append(pred)
            actual_texts.append(actual)

print(f'Accuracy: {(correct/total)*100:.1f}% ({correct}/{total})')
for i in range(5):
    print(f'\nPredicted: {predicted_texts[i]}')
    print(f'Actual: {actual_texts[i]}')

Accuracy: 0.0% (0/55)

Predicted: یییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییییی
Actual: کو نہ صرف اپنے مفادات کیلئے استعمال کر رہے ہیں بلکہ معصوم

Predicted:                                                                                                                                 
Actual: جواب میں مظاہرین نے پولیس پر پتھراؤ شروع کر دیا۔ بنوں کے

Predicted: اااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااا
Actual: پاکستانی اور انڈین شخصیات کے درمیان ٹریک ٹو ملاقاتوں اور وزرائے اعظم کو لکھے گئے خط کے حوالے سے

Predicted: اااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااا
Actual: والدین کی عزت کرنی چاہیے

Predicted: اااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااا
Actua

In [ ]:
extra_epochs = 50
for epoch in range(extra_epochs):
    model.train()
    total_loss = 0
    print(f'\nExtra Epoch {epoch + 1}/{extra_epochs}')
    print('-' * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    print(f'Extra Epoch {epoch + 1} complete | Average Loss: {total_loss/len(train_loader):.4f}')

print('\nExtra training complete!')


Extra Epoch 1/50
------------------------------
  Batch 0/55 | Loss: 3.5688
  Batch 10/55 | Loss: 3.3448
  Batch 20/55 | Loss: 3.3973
  Batch 30/55 | Loss: 3.5105
  Batch 40/55 | Loss: 3.4530
  Batch 50/55 | Loss: 3.2920
Extra Epoch 1 complete | Average Loss: 3.4408

Extra Epoch 2/50
------------------------------
  Batch 0/55 | Loss: 3.3997
  Batch 10/55 | Loss: 3.3759
  Batch 20/55 | Loss: 3.4566
  Batch 30/55 | Loss: 3.8607
  Batch 40/55 | Loss: 3.4431
  Batch 50/55 | Loss: 3.3880
Extra Epoch 2 complete | Average Loss: 3.4285

Extra Epoch 3/50
------------------------------
  Batch 0/55 | Loss: 3.3969
  Batch 10/55 | Loss: 3.3328
  Batch 20/55 | Loss: 3.3802
  Batch 30/55 | Loss: 3.4709
  Batch 40/55 | Loss: 3.3913
  Batch 50/55 | Loss: 3.5461
Extra Epoch 3 complete | Average Loss: 3.4294

Extra Epoch 4/50
------------------------------
  Batch 0/55 | Loss: 3.3412
  Batch 10/55 | Loss: 3.3144
  Batch 20/55 | Loss: 3.3364
  Batch 30/55 | Loss: 3.3059
  Batch 40/55 | Loss: 3.5166
  B

Training again model

In [ ]:
for param_group in optimizer.param_groups:
    param_group['lr'] = 2e-6  # bohat kam kar diya, 5e-5 se

extra_epochs = 10  # chhota burst, phir check karenge
for epoch in range(extra_epochs):
    model.train()
    total_loss = 0
    print(f'\nLow-LR Epoch {epoch + 1}/{extra_epochs}')
    print('-' * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch + 1} complete | Average Loss: {total_loss/len(train_loader):.4f}')


Low-LR Epoch 1/10
------------------------------
  Batch 0/55 | Loss: 3.1173
  Batch 10/55 | Loss: 3.0965
  Batch 20/55 | Loss: 3.1488
  Batch 30/55 | Loss: 2.9361
  Batch 40/55 | Loss: 3.2485
  Batch 50/55 | Loss: 2.9410
Epoch 1 complete | Average Loss: 2.9940

Low-LR Epoch 2/10
------------------------------
  Batch 0/55 | Loss: 3.0619
  Batch 10/55 | Loss: 3.1903
  Batch 20/55 | Loss: 3.0387
  Batch 30/55 | Loss: 2.8176
  Batch 40/55 | Loss: 3.1378
  Batch 50/55 | Loss: 3.0543
Epoch 2 complete | Average Loss: 2.9729

Low-LR Epoch 3/10
------------------------------
  Batch 0/55 | Loss: 2.9307
  Batch 10/55 | Loss: 3.0207
  Batch 20/55 | Loss: 3.0240
  Batch 30/55 | Loss: 2.9494
  Batch 40/55 | Loss: 2.7362
  Batch 50/55 | Loss: 3.1076
Epoch 3 complete | Average Loss: 2.9522

Low-LR Epoch 4/10
------------------------------
  Batch 0/55 | Loss: 2.8490
  Batch 10/55 | Loss: 2.8111
  Batch 20/55 | Loss: 2.9888
  Batch 30/55 | Loss: 2.9607
  Batch 40/55 | Loss: 3.0048
  Batch 50/55 | L

Evaluation checking

In [ ]:
model.eval()
correct = 0
total = 0
predicted_texts, actual_texts = [], []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=128,
            repetition_penalty=1.3,
            no_repeat_ngram_size=2
        )
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)
        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            predicted_texts.append(pred)
            actual_texts.append(actual)
print(f'Accuracy: {(correct/total)*100:.1f}% ({correct}/{total})')
for i in range(5):
    print(f'\nPredicted: {predicted_texts[i]}')
    print(f'Actual: {actual_texts[i]}')

Accuracy: 0.0% (0/55)

Predicted: میکلعہناجپبتشقوآ مےغا�گرفحںھچخز دئ’ط �ظسصض الذ�ثَ" �ٰ،�ڈ isٹُْ۔ڑ۳؟�9‘ّ�
Actual: کو نہ صرف اپنے مفادات کیلئے استعمال کر رہے ہیں بلکہ معصوم

Predicted: کشاقیمرہجنپتگآخےدو فبغ’لچا�ئں �ح مظزسعصطڈھٹ�ُ �:-َذ9۹۳ض14۔ْ‘ث؟�۱� ال is�ڑ
Actual: جواب میں مظاہرین نے پولیس پر پتھراؤ شروع کر دیا۔ بنوں کے

Predicted: اپجکنبی’ہرےخ شولستگچ �آفطا�ٹمڑھئزقڈص"حغظ14َں�عد۔ذ9ثض؟‘ال ٪ُْmethod 15 3�ؤ م�
Actual: پاکستانی اور انڈین شخصیات کے درمیان ٹریک ٹو ملاقاتوں اور وزرائے اعظم کو لکھے گئے خط کے حوالے سے

Predicted: پاآیجموکخبقہفشرص’لحچئغگتز9نےں ط مدضڈا�ٹث14ُس۱ھَ۳۹ �۷‘،؟ذ:-ء۵ظع ال۔Complال isCL
Actual: والدین کی عزت کرنی چاہیے

Predicted: امینجکوہقآپ’فشئلخگغتںےب ظصز9چڈر م �د14ٹحعس،ھ‘ذضطث۳� الا�َ۔۹314�:-"Thanks limitedال�ُٰ
Actual: فرانس نے منگولیا کو 7 کروڑ سال پرانا ڈائناسور کا ڈھانچہ واپس کر دیا
